# 🎓 MY-AI ট্রেনিং নোটবুক (Google Colab)

এই নোটবুক দিয়ে আপনি **Hugging Face-এর যেকোনো ডাটাসেট** (বা আপনার নিজের এক্সপোর্ট করা ডাটা) দিয়ে একটি ওপেন-সোর্স মডেলকে ফাইন-টিউন করবেন।

**শুরু করার আগে:**
1. `Runtime → Change runtime type → T4 GPU` সিলেক্ট করুন
2. উপরের সেলগুলো ক্রমানুসারে চালান

**ফ্লো:**
```
Hugging Face ডাটাসেট → build_from_hf.py → train.jsonl/val.jsonl
                                              ↓
                              train_lora.py (LoRA ফাইন-টিউন)
                                              ↓
                              my-ai-model (GGUF, Ollama-রেডি)
```

In [ ]:
# ✅ GPU আছে কিনা দেখুন — শেষে "Tesla T4" দেখালে OK
!nvidia-smi

In [ ]:
# 📦 দরকারি প্যাকেজ ইনস্টল (২-৩ মিনিট লাগে)
!pip install -q unsloth "unsloth[colab-new]" datasets transformers trl
print("✅ ইনস্টল শেষ")

In [ ]:
# 📤 রিপোর training/ ফোল্ডার থেকে এই ২টা ফাইল আপলোড করুন:
#    - build_from_hf.py   (Hugging Face ডাটাসেট কনভার্টার)
#    - train_lora.py      (LoRA ট্রেনিং স্ক্রিপ্ট)
# নিজের ডাটা ব্যবহার করলে build_dataset.py-ও দিন
from google.colab import files
print("ফাইলগুলো বাছাই করুন (Ctrl/Cmd দিয়ে একাধিক)...")
uploaded = files.upload()
print("\n✅ আপলোড শেষ:", list(uploaded.keys()))

---

## 🅰️ অপশন A — Hugging Face ডাটাসেট দিয়ে ট্রেন

নিচের সেলে ডাটাসেটের নাম/সাইজ বদলাতে পারেন। জনপ্রিয় চ্যাট ডাটাসেট:

| ডাটাসেট | কী শেখায় | সাইজ টিপস |
|---|---|---|
| `HuggingFaceH4/ultrachat_200k` (split `train_sft`) | সাধারণ চ্যাট | `--limit 5000` দিয়ে শুরু করুন |
| `Open-Orca/OpenOrca` | প্রশ্ন-উত্তর, রিজনিং | `--limit 5000` |
| `tatsu-lab/alpaca` | ইনস্ট্রাকশন ফলোয়িং | ছোট, পুরোটাই নিতে পারেন |

> যেকোনো ShareGPT-স্টাইল (`messages` ফিল্ড) ডাটাসেট চলবে। প্লেইন `prompt`/`response` কলাম থাকলে `--prompt-field` ও `--response-field` ব্যবহার করুন।

In [ ]:
# 🔄 হাগিং ফেস ডাটাসেট → train.jsonl / val.jsonl
!python build_from_hf.py \
    --dataset HuggingFaceH4/ultrachat_200k \
    --split train_sft \
    --limit 5000 \
    --streaming \
    --output data

# নিজের PC/Colab-এ জায়গা বেশি থাকলে --streaming বাদ দিতে পারেন।
# ছোট ডাটাসেট (alpaca) হলে: --dataset tatsu-lab/alpaca --split train --limit 0

---

## 🅱️ অপশন B — নিজের এক্সপোর্ট করা ডাটা দিয়ে ট্রেন

MY-AI ড্যাশবোর্ড → **AI Brain** → **Export Dataset** → `myai-dataset.jsonl` ডাউনলোড করে নিচের সেলে আপলোড করুন।
চাইলে HF ডাটার সাথে নিজের ডাটাও মিশিয়ে নিতে পারেন — দুটোর ফরম্যাট একই, `train.jsonl` ফাইলের নিচে নিজের লাইনগুলো জুড়ে দিলেই হবে।

In [ ]:
# 📤 নিজের ডাটা আপলোড + কনভার্ট (শুধু অপশন B ব্যবহার করলে চালান)
from google.colab import files
uploaded = files.upload()  # myai-dataset.jsonl
!python build_dataset.py --input myai-dataset.jsonl --output data

---

## 🚀 ট্রেনিং (LoRA ফাইন-টিউন)

ফ্রি Colab T4 GPU-তে প্রায় **১০-১৫ মিনিট** লাগে (৫ হাজার ডাটা, ১-২ epoch)।

**ফ্রি GPU কোথায় পাবেন (একদম টাকা লাগে না):**
| প্ল্যাটফর্ম | ফ্রি কোটা | নোট |
|---|---|---|
| Google Colab | T4 GPU, ~১২ ঘণ্টার সেশন | সবচেয়ে সহজ ✅ |
| Kaggle | সপ্তাহে ~৩০ ঘণ্টা GPU, 2x T4 | Colab-এর লিমিট শেষ হলে এখানে চালান |

💡 **প্রো টিপ:** Colab-এ সেশন/লিমিট শেষ হলে একই ফাইল **Kaggle notebook**-এ আপলোড করে চালিয়ে দিন — দুই জায়গা মিলিয়ে ২৪ ঘণ্টায় অনেক ট্রেনিং ফ্রি-তে করা যায়।

**মডেল বাছাই:**
| মডেল | VRAM | নোট |
|---|---|---|
| `sarvamai/sarvam-1` | ~4-6 GB | 🇧🇩 বাংলা-সহ ১০টি ভারতীয় ভাষা + ইংরেজি — বাংলা চ্যাটের জন্য সেরা ✅ |
| `unsloth/Llama-3.2-1B-Instruct` | ~4-6 GB | সবচেয়ে নিরাপদ, T4-তে ভালো চলে, ইংরেজিতে শক্ত |
| `unsloth/Qwen2.5-1.5B-Instruct` | ~5-7 GB | বহুভাষী (বাংলাও কিছুটা) |
| `unsloth/gemma-2-2b-it` | ~6-8 GB | ভালো মান |

> 🇧🇩 **বাংলা+ইংরেজি মিক্সড চ্যাট চাইলে:** `sarvamai/sarvam-1` (Apache 2.0, Llama আর্কিটেকচার) বাছুন — বাংলা-সহ ১০টি ভারতীয় ভাষায় ট্রেন করা, ২B সাইজ বলে ফ্রি T4-তে অনায়াসে চলে। Unsloth-এ লোড করতে কোনো সমস্যা হলে fallback: `unsloth/Qwen2.5-3B-Instruct`।

In [ ]:
# 🚀 ট্রেন চালু!
!python train_lora.py \
    --model unsloth/Llama-3.2-1B-Instruct \
    --train data/train.jsonl \
    --val data/val.jsonl \
    --output ./my-ai-model \
    --epochs 2 \
    --export-gguf

# বেশি ডাটা (২০ হাজার+) থাকলে: --epochs 1
# শক্তিশালী মডেল চাইলে: --model unsloth/Qwen2.5-1.5B-Instruct

In [ ]:
# ⬇️ ট্রেন করা মডেল ডাউনলোড করুন (zip আকারে)
!zip -r -q my-ai-model.zip my-ai-model
from google.colab import files
files.download('my-ai-model.zip')
print("✅ ডাউনলোড শুরু হয়েছে — GGUF ফাইলটা নিজের PC-তে রাখুন")
print("👉 এরপর: ollama create my-ai -f Modelfile && ollama run my-ai")
print("   (বিস্তারিত: training/README.md-এর ধাপ ৪ ও ৫)")

## 💡 সত্যি কথাটা

ছোট ডাটায় ফাইন-টিউন করলে মডেল আপনার **স্টাইল/ফরম্যাট** শিখবে, কিন্তু বিশ্ব-জ্ঞান বাড়বে না। নতুন ফ্যাক্ট/ডকুমেন্ট শেখাতে চাইলে MY-AI-এর **Knowledge ট্যাব (RAG)** ব্যবহার করুন — সেটাই সবচেয়ে কার্যকর।

দুটো একসাথে ব্যবহার করলে সেরা: **LoRA = আপনার ঢং**, **RAG = আপনার তথ্য**।